# Hindi Corpus Finalisation — Balanced Split & Tokenizer (Model H)

Run this **after** `scripts/collect_hindi_manual_web.py` has finished pushing the
new `manual_web_shard_*.jsonl` files to Drive.

What this notebook fixes relative to the earlier split cell:

1. **Measured chars-per-token instead of a single guess.** The old cell used
   `CHARS_PER_TOKEN = 3.19` for both partitions. Measured with the actual
   `hindi_bpe_8k` model, manual text is **3.320** chars/token and Sangraha text
   is **3.516**. Using 3.19 for both overstated the token count, which is why a
   run that "reached 400M automated tokens" produced materially fewer.
2. **The automated budget is derived from the manual total**, so the >= 20%
   manual floor and the >= 500M total are enforced together rather than hoped for.
3. **Hash-based split routing** — a document's split is a function of its
   content hash, so it is stable no matter what order shards are ingested in.
4. **Manual-priority deduplication at split time**, so a Sangraha document can
   never displace a manually collected one.


---
## Cell 0: Setup & Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q sentencepiece

import gc, hashlib, json, os, shutil, time
from collections import Counter, defaultdict
from pathlib import Path

import sentencepiece as spm

LANGUAGE      = 'hindi'
DRIVE_BASE    = '/content/drive/MyDrive/24KaratAu/hindi'
MANUAL_DIR    = f'{DRIVE_BASE}/cleaned/manual'
NONWEB_DIR    = f'{DRIVE_BASE}/cleaned/automated_nonweb'
SPLITS_DIR    = f'{DRIVE_BASE}/splits'
TOKENIZER_DIR = f'{DRIVE_BASE}/tokenizer'
MANIFEST_DIR  = f'{DRIVE_BASE}/manifests'

for d in (SPLITS_DIR, TOKENIZER_DIR, MANIFEST_DIR):
    os.makedirs(d, exist_ok=True)

# Multi-GB reads over the Drive FUSE mount drop sockets; stage on local NVMe.
LOCAL_MANUAL = '/content/local_manual'
LOCAL_AUTO   = '/content/local_auto'
LOCAL_SPLITS = '/content/splits'
for d in (LOCAL_MANUAL, LOCAL_AUTO, LOCAL_SPLITS):
    os.makedirs(d, exist_ok=True)

# Fallbacks only. Cell 1b re-measures both figures against the current
# tokenizer, because they are not stable across sources: the Wikipedia/literary
# shards tokenise at 3.320 chars/token, the Colab news shards at 3.387 and the
# sitemap web crawl at 3.538, blending to 3.396 over the manual partition.
# Budgeting off one stale constant is what caused the earlier shortfall.
MANUAL_CHARS_PER_TOKEN = 3.396
AUTO_CHARS_PER_TOKEN   = 3.516

TOTAL_TOKEN_TARGET = 520_000_000   # comfortably clears the 500M floor
MIN_MANUAL_SHARE   = 0.20
SPLIT_RATIOS       = {'train': 0.90, 'val': 0.05, 'test': 0.05}
SPLIT_SALT         = '42'          # documented split seed

print('Manual shards   :', len(list(Path(MANUAL_DIR).glob('*.jsonl'))))
print('Automated shards:', len(list(Path(NONWEB_DIR).glob('*.jsonl'))))

---
## Cell 1: Stage shards on local disk

In [ ]:
t0 = time.time()

def _remount():
    """Recover a dropped Drive FUSE mount."""
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(3)
    drive.mount('/content/drive', force_remount=True)
    time.sleep(3)


def _list_drive_shards(src_dir, attempts=4):
    for _ in range(attempts):
        try:
            return {q.name: q.stat().st_size
                    for q in sorted(Path(src_dir).glob('*.jsonl'))}
        except OSError as exc:
            print(f'  listing {src_dir} failed ({exc}); remounting...')
            _remount()
    raise RuntimeError(f'Could not list {src_dir}')


def stage(src_dir, dst_dir, label, attempts=4):
    """Copy shards one at a time, verifying each size against Drive.

    A single bulk `cp` of 220 files over the Drive FUSE mount silently loses
    files when the transport drops — and `cp ... 2>/dev/null` hides it. The
    first run of this notebook staged 2 of 220 automated shards and produced a
    corpus that was 95.7% manual while every printed check still looked fine.
    Copying per file, comparing sizes, and remounting on failure makes that
    failure mode loud instead of silent.
    """
    expected = _list_drive_shards(src_dir)
    dst = Path(dst_dir)
    missing = []
    print(f'{label}: {len(expected)} shards on Drive')
    for i, (name, size) in enumerate(expected.items(), 1):
        target = dst / name
        if target.exists() and target.stat().st_size == size:
            continue
        for attempt in range(attempts):
            try:
                shutil.copyfile(Path(src_dir) / name, target)
                got = target.stat().st_size
                if got != size:
                    raise OSError(f'size mismatch: {got:,} != {size:,}')
                break
            except OSError as exc:
                print(f'  {name}: attempt {attempt + 1}/{attempts} failed ({exc})')
                if target.exists():
                    target.unlink()
                _remount()
        else:
            missing.append(name)
        if i % 25 == 0:
            print(f'  {label}: {i}/{len(expected)} staged')
    staged = sorted(dst.glob('*.jsonl'))
    print(f'{label}: staged {len(staged)}/{len(expected)} shards'
          + (f' — MISSING {missing}' if missing else ''))
    return staged, expected, missing


manual_files, manual_expected, manual_missing = stage(MANUAL_DIR, LOCAL_MANUAL, 'manual')
auto_files, auto_expected, auto_missing = stage(NONWEB_DIR, LOCAL_AUTO, 'automated')
print(f'Staged {len(manual_files)} manual + {len(auto_files)} automated shards '
      f'in {time.time() - t0:.0f}s')

# Manual accounting must be exact: every manual shard has to be present.
assert not manual_missing, f'Manual shards missing after retries: {manual_missing}'
assert len(auto_files) >= 0.98 * len(auto_expected), (
    f'Only {len(auto_files)} of {len(auto_expected)} automated shards staged. '
    f'Re-run this cell — it resumes from what is already on local disk.')


def source_group(name: str) -> str:
    """Bucket a shard filename into the source categories used in the report."""
    if name.startswith('wiki_shard'):        return 'Wikipedia dump'
    if name.startswith('wikisource'):        return 'Wikisource dump'
    if name.startswith('manual_wikisource'): return 'Wikisource dump'
    if name.startswith('manual_lit'):        return 'Literature dump'
    if name.startswith('manual_aux'):        return 'Wikibooks / Wikiquote'
    if name.startswith('manual_news'):       return 'News crawl (Colab)'
    if name.startswith('manual_web'):        return 'Web crawl (sitemap)'
    if name.startswith('manual_'):           return 'Other manual'
    return 'Automated (Sangraha)'


def read_jsonl(path):
    """Yield docs, tolerating the truncated lines a dropped FUSE write can leave."""
    bad = 0
    with open(path, 'r', encoding='utf-8') as fh:
        for line in fh:
            if not line.strip():
                continue
            try:
                yield json.loads(line)
            except (json.JSONDecodeError, UnicodeDecodeError):
                bad += 1
    if bad:
        print(f'  skipped {bad} corrupt lines in {Path(path).name}')


---
## Cell 1b: Re-measure chars-per-token against the current tokenizer

In [ ]:
import random as _random

EXISTING_TOKENIZER = f'{TOKENIZER_DIR}/hindi_bpe_8k.model'

def measure_chars_per_token(files, sp_model, sample_rate=0.02, max_docs=20_000):
    """Sample documents across shards and tokenise them for a real ratio.

    Returns None when no tokenizer exists yet, so the caller keeps its fallback.
    """
    if not os.path.exists(sp_model):
        return None
    sp = spm.SentencePieceProcessor()
    sp.load(sp_model)
    rng = _random.Random(42)
    chars = tokens = seen = 0
    for path in files:
        for doc in read_jsonl(path):
            text = doc.get('text', '')
            if not text.strip() or rng.random() >= sample_rate:
                continue
            chars  += len(text)
            tokens += len(sp.encode(text))
            seen   += 1
            if seen >= max_docs:
                break
        if seen >= max_docs:
            break
    return chars / tokens if tokens else None

measured_manual = measure_chars_per_token(manual_files, EXISTING_TOKENIZER)
measured_auto   = measure_chars_per_token(auto_files,   EXISTING_TOKENIZER)

if measured_manual:
    print(f'Manual    : {MANUAL_CHARS_PER_TOKEN:.3f} (default) -> '
          f'{measured_manual:.3f} (measured)')
    MANUAL_CHARS_PER_TOKEN = measured_manual
if measured_auto:
    print(f'Automated : {AUTO_CHARS_PER_TOKEN:.3f} (default) -> '
          f'{measured_auto:.3f} (measured)')
    AUTO_CHARS_PER_TOKEN = measured_auto
if not measured_manual:
    print(f'No tokenizer at {EXISTING_TOKENIZER}; keeping documented defaults.')

---
## Cell 2: Manual inventory & automated budget

In [ ]:
manual_chars = 0
manual_docs  = 0
manual_dupes = 0
manual_dupe_chars = 0
by_group     = defaultdict(lambda: {'docs': 0, 'chars': 0, 'shards': 0})

# Deduplicate the manual partition here, not just against the automated stream.
# The shards were written by several collection runs whose dedup registries were
# snapshotted at different times, so the same article can appear twice under two
# URLs. Counting duplicates into manual_chars would also inflate the automated
# budget derived from it.
manual_hashes = set()

for path in manual_files:
    group = source_group(path.name)
    by_group[group]['shards'] += 1
    for doc in read_jsonl(path):
        text = doc.get('text', '')
        if not text.strip():
            continue
        h = doc.get('text_sha256') or hashlib.sha256(text.encode('utf-8')).hexdigest()
        key = int(h[:16], 16)
        if key in manual_hashes:
            manual_dupes += 1
            manual_dupe_chars += len(text)
            continue
        manual_hashes.add(key)
        manual_docs  += 1
        manual_chars += len(text)
        by_group[group]['docs']  += 1
        by_group[group]['chars'] += len(text)

manual_tokens = manual_chars / MANUAL_CHARS_PER_TOKEN

# manual_share = M / (M + A) >= 0.20  <=>  A <= 4M
max_auto_by_share = manual_tokens * (1 - MIN_MANUAL_SHARE) / MIN_MANUAL_SHARE
auto_token_budget = min(max_auto_by_share, TOTAL_TOKEN_TARGET - manual_tokens)
auto_token_budget = max(auto_token_budget, 0)
auto_char_budget  = int(auto_token_budget * AUTO_CHARS_PER_TOKEN)

projected_total = manual_tokens + auto_token_budget
projected_share = manual_tokens / projected_total if projected_total else 0

print(f'{"Manual source":<26}{"Shards":>8}{"Docs":>12}{"Chars":>16}{"Tokens":>14}')
for group, s in sorted(by_group.items(), key=lambda kv: -kv[1]['chars']):
    print(f'{group:<26}{s["shards"]:>8}{s["docs"]:>12,}{s["chars"]:>16,}'
          f'{int(s["chars"] / MANUAL_CHARS_PER_TOKEN):>14,}')
print('-' * 76)
print(f'{"TOTAL MANUAL":<26}{len(manual_files):>8}{manual_docs:>12,}'
      f'{manual_chars:>16,}{int(manual_tokens):>14,}')
print(f'Duplicate manual documents dropped: {manual_dupes:,} '
      f'({manual_dupe_chars:,} chars)')
print()
print(f'Automated token budget : {int(auto_token_budget):,} '
      f'({int(auto_char_budget):,} chars)')
print(f'Projected total        : {int(projected_total):,} tokens')
print(f'Projected manual share : {projected_share * 100:.2f}%')

assert projected_total >= 500_000_000, (
    f'Only {int(projected_total):,} tokens reachable. Collect more manual data: '
    f'every manual token unlocks four automated ones.')
assert projected_share >= MIN_MANUAL_SHARE, 'Manual floor not met.'

---
## Cell 3: Streaming 90/5/5 split

In [ ]:
def split_for(doc_hash: str) -> str:
    """Route a document by content hash, so the split is order-independent."""
    bucket = int(hashlib.sha256((SPLIT_SALT + doc_hash).encode()).hexdigest()[:8], 16) % 10_000
    if bucket < 9_000:
        return 'train'
    return 'val' if bucket < 9_500 else 'test'

txt  = {s: open(f'{LOCAL_SPLITS}/{s}.txt',   'w', encoding='utf-8') for s in SPLIT_RATIOS}
jsnl = {s: open(f'{LOCAL_SPLITS}/{s}.jsonl', 'w', encoding='utf-8') for s in SPLIT_RATIOS}

counts       = Counter()
split_chars  = Counter()
manual_seen  = set()     # 64-bit content hashes, manual takes precedence
written_manual_chars = 0
written_auto_chars   = 0
auto_dropped_dupes   = 0

def emit(doc, text, is_manual):
    h = doc.get('text_sha256') or hashlib.sha256(text.encode('utf-8')).hexdigest()
    s = split_for(h)
    doc = dict(doc, collection_type='manual' if is_manual else 'automated')
    jsnl[s].write(json.dumps(doc, ensure_ascii=False) + '\n')
    txt[s].write(text.strip() + '\n\n')
    counts[s] += 1
    split_chars[s] += len(text)
    return h

print('Pass 1 — all unique manual documents...')
manual_dupes_dropped = 0
for path in manual_files:
    for doc in read_jsonl(path):
        text = doc.get('text', '')
        if not text.strip():
            continue
        h = doc.get('text_sha256') or hashlib.sha256(text.encode('utf-8')).hexdigest()
        key = int(h[:16], 16)
        if key in manual_seen:
            manual_dupes_dropped += 1
            continue
        manual_seen.add(key)
        emit(doc, text, True)
        written_manual_chars += len(text)
print(f'  {sum(counts.values()):,} manual docs, {written_manual_chars:,} chars '
      f'({manual_dupes_dropped:,} duplicates dropped)')

print(f'Pass 2 — automated up to {auto_char_budget:,} chars...')
for path in auto_files:
    if written_auto_chars >= auto_char_budget:
        break
    for doc in read_jsonl(path):
        if written_auto_chars >= auto_char_budget:
            break
        text = doc.get('text', '')
        if not text.strip():
            continue
        h = doc.get('text_sha256') or hashlib.sha256(text.encode('utf-8')).hexdigest()
        if int(h[:16], 16) in manual_seen:
            auto_dropped_dupes += 1
            continue
        emit(doc, text, False)
        written_auto_chars += len(text)
    gc.collect()

for fh in list(txt.values()) + list(jsnl.values()):
    fh.close()

est_manual_tokens = written_manual_chars / MANUAL_CHARS_PER_TOKEN
est_auto_tokens   = written_auto_chars / AUTO_CHARS_PER_TOKEN
est_total         = est_manual_tokens + est_auto_tokens

print('=' * 68)
print(f'Manual    : {written_manual_chars:>16,} chars  ~{int(est_manual_tokens):>13,} tokens'
      f'  {est_manual_tokens / est_total * 100:5.2f}%')
print(f'Automated : {written_auto_chars:>16,} chars  ~{int(est_auto_tokens):>13,} tokens'
      f'  {est_auto_tokens / est_total * 100:5.2f}%')
print(f'TOTAL     : {written_manual_chars + written_auto_chars:>16,} chars  '
      f'~{int(est_total):>13,} tokens')
print(f'Automated docs dropped as manual duplicates: {auto_dropped_dupes:,}')
for s in SPLIT_RATIOS:
    print(f'  {s:<6} {counts[s]:>10,} docs  {split_chars[s]:>16,} chars')
print('=' * 68)

# Assert on the OUTCOME, not the plan, and only after the numbers are printed.
# Cell 2's assert merely checks the budget is reachable; it stays green even if
# Pass 2 finds nothing to read. These three are what stop a bad corpus from
# reaching Drive.
assert written_auto_chars >= 0.98 * auto_char_budget, (
    f'Automated ingest reached only {written_auto_chars:,} of the '
    f'{auto_char_budget:,}-char budget ({auto_char_budget - written_auto_chars:,} '
    f'short) from {len(auto_files)} staged shards. The automated shards are not '
    f'fully staged on local disk — re-run Cell 1, which resumes where it left off.')
assert est_total >= 500_000_000, f'Only {int(est_total):,} tokens produced.'
assert est_manual_tokens / est_total >= MIN_MANUAL_SHARE, 'Manual floor breached.'

# Copy each split individually and verify its size on Drive afterwards. A bulk
# `cp` of the ~3.9 GB train.txt across the FUSE mount fails silently: the other
# files still land, so the folder looks complete while train.txt is left over
# from a previous run. Retry with a remount, exactly as staging does.
print('Syncing splits to Drive...')

def sync_to_drive(name, attempts=4):
    src, dst = f'{LOCAL_SPLITS}/{name}', f'{SPLITS_DIR}/{name}'
    want = os.path.getsize(src)
    for attempt in range(attempts):
        try:
            shutil.copyfile(src, dst)
            got = os.path.getsize(dst)
            if got == want:
                print(f'  {name}: {got:,} bytes OK')
                return True
            raise OSError(f'size mismatch on Drive: {got:,} != {want:,}')
        except OSError as exc:
            print(f'  {name}: attempt {attempt + 1}/{attempts} failed ({exc})')
            _remount()
    print(f'  {name}: FAILED after {attempts} attempts')
    return False

synced = [sync_to_drive(n) for n in
          ('train.txt', 'val.txt', 'test.txt',
           'train.jsonl', 'val.jsonl', 'test.jsonl')]
assert all(synced), 'A split did not reach Drive intact — re-run this sync.'
print('Done.')

---
## Cell 4: Retrain the 8k BPE tokenizer on the new train split

In [ ]:
MPREFIX   = f'{TOKENIZER_DIR}/hindi_bpe_8k'
TRAIN_TXT = f'{LOCAL_SPLITS}/train.txt'
print(f'Training BPE 8k on {os.path.getsize(TRAIN_TXT) / 1024**2:.1f} MB...')

spm.SentencePieceTrainer.train(
    input=TRAIN_TXT, model_prefix='/content/hindi_bpe_8k', vocab_size=8000,
    model_type='bpe', character_coverage=0.9995, num_threads=8,
    max_sentence_length=8192, input_sentence_size=1_000_000,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    pad_piece='<pad>', unk_piece='<unk>', bos_piece='<bos>', eos_piece='<eos>',
    byte_fallback=True,
    split_by_unicode_script=True, split_by_whitespace=True, split_digits=True,
)
!cp /content/hindi_bpe_8k.model /content/hindi_bpe_8k.vocab {TOKENIZER_DIR}/
print('Tokenizer saved to Drive.')

---
## Cell 5: Exact token counts + report table

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load('/content/hindi_bpe_8k.model')

def count_tokens(jsonl_path, sample_rate=1.0):
    """Exact token count over a split (sample_rate < 1 extrapolates by characters)."""
    import random as _r
    rng = _r.Random(0)
    tokens = chars = sampled_chars = docs = 0
    for doc in read_jsonl(jsonl_path):
        text = doc.get('text', '')
        if not text.strip():
            continue
        docs += 1
        chars += len(text)
        if sample_rate >= 1.0 or rng.random() < sample_rate:
            tokens += len(sp.encode(text))
            sampled_chars += len(text)
    if sampled_chars and sampled_chars < chars:
        tokens = int(tokens * chars / sampled_chars)
    return docs, chars, tokens

rows, grand = [], [0, 0, 0]
for split in ('train', 'val', 'test'):
    d, c, t = count_tokens(f'{LOCAL_SPLITS}/{split}.jsonl', sample_rate=0.10)
    rows.append((split, d, c, t))
    grand = [grand[0] + d, grand[1] + c, grand[2] + t]

print(f'{"Split":<8}{"Docs":>12}{"Chars":>18}{"Tokens":>16}{"Share":>9}')
for split, d, c, t in rows:
    print(f'{split:<8}{d:>12,}{c:>18,}{t:>16,}{t / grand[2] * 100:>8.1f}%')
print('-' * 63)
print(f'{"TOTAL":<8}{grand[0]:>12,}{grand[1]:>18,}{grand[2]:>16,}')
print()
print(f'Measured chars/token on the final corpus: {grand[1] / grand[2]:.3f}')

summary = {
    'generated': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'manual_chars': written_manual_chars,
    'automated_chars': written_auto_chars,
    'manual_tokens_est': int(est_manual_tokens),
    'automated_tokens_est': int(est_auto_tokens),
    'manual_share_pct': round(est_manual_tokens / est_total * 100, 2),
    'total_tokens_measured': grand[2],
    'splits': {s: {'docs': d, 'chars': c, 'tokens': t} for s, d, c, t in rows},
    'manual_sources': {g: dict(v) for g, v in by_group.items()},
}
with open(f'{MANIFEST_DIR}/hindi_corpus_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)
print(f'Wrote {MANIFEST_DIR}/hindi_corpus_summary.json')